In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

df = pd.read_csv("../data/raw/retail_customer_churn_100k (1).csv")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

df.head()

Dataset loaded successfully!
Dataset shape: (100000, 18)


,customer_id,age_group,gender,region,customer_segment,preferred_channel,purchase_frequency,avg_order_value,total_spent,recency_days,website_visits,discount_usage_rate,email_open_rate,cart_abandonment_rate,loyalty_score,engagement_score,churn_risk,churn_flag
0,C000001,18-24,Other,North,Loyal,Online,5,381.27,2108.80,189,27,0.38,0.13,0.42,33,35.5,Medium,1
1,C000002,25-34,Other,South,New,Mobile App,12,376.65,3843.06,277,68,0.51,0.06,0.07,68,49.5,Medium,1
2,C000003,55+,Other,North,VIP,Online,5,37.73,224.65,271,23,0.64,0.22,0.46,73,38.0,High,1
3,C000004,18-24,Male,North,Returning,Mobile App,12,191.19,1960.79,89,93,0.66,0.31,0.13,93,59.0,Low,0
4,C000005,55+,Other,North,Returning,Mobile App,2,279.64,594.79,264,4,0.73,0.27,0.58,11,34.5,High,1


In [5]:
print("Columns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nChurn distribution:")
print(df["churn_flag"].value_counts())

Columns:
['customer_id', 'age_group', 'gender', 'region', 'customer_segment', 'preferred_channel', 'purchase_frequency', 'avg_order_value', 'total_spent', 'recency_days', 'website_visits', 'discount_usage_rate', 'email_open_rate', 'cart_abandonment_rate', 'loyalty_score', 'engagement_score', 'churn_risk', 'churn_flag']

Missing values:
customer_id              0
age_group                0
gender                   0
region                   0
customer_segment         0
preferred_channel        0
purchase_frequency       0
avg_order_value          0
total_spent              0
recency_days             0
website_visits           0
discount_usage_rate      0
email_open_rate          0
cart_abandonment_rate    0
loyalty_score            0
engagement_score         0
churn_risk               0
churn_flag               0
dtype: int64

Churn distribution:
churn_flag
0    53776
1    46224
Name: count, dtype: int64


In [7]:
from pathlib import Path
import sys

project_root = Path.cwd()
while not (project_root / "src").is_dir() and project_root.parent != project_root:
    project_root = project_root.parent
if not (project_root / "src").is_dir():
    raise FileNotFoundError("Could not locate the project root containing src/.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from sklearn.model_selection import train_test_split
from src.config import MODEL_OUTPUT_PATH, PROCESSED_DATA_PATH, RANDOM_STATE, TEST_SIZE
from src.data_processor import prepare_model_data, split_features_target

model_df = prepare_model_data(df)
X, y = split_features_target(model_df)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

best_model = joblib.load(MODEL_OUTPUT_PATH)
best_model_name = type(best_model.named_steps["model"]).__name__
y_pred = best_model.predict(X_test)
model_step = best_model.named_steps["model"]

if hasattr(model_step, "feature_importances_"):

    importances = model_step.feature_importances_

    print("Feature importance available.")

else:

    print(
        "The selected model does not provide "
        "feature_importances_ directly."
    )

The selected model does not provide feature_importances_ directly.


In [8]:
preprocessor_fitted = (
    best_model
    .named_steps["preprocessor"]
)

feature_names = (
    preprocessor_fitted
    .get_feature_names_out()
)

print("Number of processed features:")
print(len(feature_names))

Number of processed features:
30


In [9]:
if hasattr(model_step, "feature_importances_"):

    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": importances
    })

    importance_df = importance_df.sort_values(
        "Importance",
        ascending=False
    )

    importance_df.head(20)

In [10]:
def churn_rate_table(data, column):

    result = (
        data.groupby(column)["churn_flag"]
        .agg(
            Customers="count",
            Churn_Rate="mean"
        )
        .sort_values(
            "Churn_Rate",
            ascending=False
        )
    )

    result["Churn_Rate"] *= 100

    return result.round(2)

In [11]:
churn_rate_table(
    df,
    "customer_segment"
)

,Customers,Churn_Rate
customer_segment,,
VIP,24906,46.67
New,25113,46.32
Returning,25106,45.98
Loyal,24875,45.93


In [12]:
churn_rate_table(
    df,
    "preferred_channel"
)

,Customers,Churn_Rate
preferred_channel,,
Online,33430,46.66
In-Store,33303,46.26
Mobile App,33267,45.75


In [13]:
churn_rate_table(
    df,
    "region"
)

,Customers,Churn_Rate
region,,
West,20039,46.50
Central,20012,46.45
East,20240,46.36
North,20035,46.33
South,19674,45.46


In [15]:
churn_rate_table(
    df,
    "age_group"
)

,Customers,Churn_Rate
age_group,,
55+,19882,46.51
35-44,20109,46.37
18-24,19987,46.24
45-54,20093,46.11
25-34,19929,45.89


In [16]:
test_results = X_test.copy()

test_results["actual_churn"] = y_test.values
test_results["predicted_churn"] = y_pred

test_results.head()

,age_group,gender,region,customer_segment,preferred_channel,purchase_frequency,avg_order_value,total_spent,recency_days,website_visits,discount_usage_rate,email_open_rate,cart_abandonment_rate,loyalty_score,engagement_score,actual_churn,predicted_churn
25119,25-34,Male,West,Loyal,Online,12,106.85,1462.91,131,67,0.23,0.24,0.57,14,33.5,1,1
84952,45-54,Male,East,Returning,Online,35,35.79,1440.11,242,240,0.14,0.10,0.36,80,37.0,0,0
37029,18-24,Female,West,Loyal,Mobile App,20,201.71,3737.23,60,153,0.73,0.74,0.25,22,74.5,0,0
32013,35-44,Male,East,New,In-Store,20,250.27,5417.70,109,65,0.52,0.06,0.11,40,47.5,0,0
86566,18-24,Male,North,VIP,In-Store,5,318.85,1450.92,18,13,0.51,0.70,0.17,59,76.5,0,0


In [17]:
processed_data = model_df.copy()

processed_data_path = Path(PROCESSED_DATA_PATH)
processed_data_path.parent.mkdir(parents=True, exist_ok=True)
processed_data.to_csv(processed_data_path, index=False)

print(
    "Processed dataset saved successfully."
)

Processed dataset saved successfully.


In [18]:
joblib.dump(
    best_model,
    MODEL_OUTPUT_PATH
)

print(
    f"Saved model: {MODEL_OUTPUT_PATH}"
)

Saved model: c:\Users\USER\Desktop\AWS_Project\models\best_customer_churn_model.pkl


In [19]:
model_information = {
    "model_name": best_model_name,
    "target": "churn_flag",
    "excluded_features": [
        "customer_id",
        "churn_risk"
    ],
    "training_rows": len(X_train),
    "testing_rows": len(X_test),
    "selection_metric": "CV ROC-AUC"
}

model_information

{'model_name': 'LogisticRegression',
 'target': 'churn_flag',
 'excluded_features': ['customer_id', 'churn_risk'],
 'training_rows': 80000,
 'testing_rows': 20000,
 'selection_metric': 'CV ROC-AUC'}

In [20]:
def predict_customer_churn(
    customer_data,
    model=best_model
):

    prediction = model.predict(
        customer_data
    )

    probability = model.predict_proba(
        customer_data
    )[:, 1]

    result = customer_data.copy()

    result["predicted_churn"] = prediction
    result["churn_probability"] = probability

    return result

In [21]:
sample_customer = X_test.iloc[[0]]

prediction_result = predict_customer_churn(
    sample_customer
)

prediction_result

,age_group,gender,region,customer_segment,preferred_channel,purchase_frequency,avg_order_value,total_spent,recency_days,website_visits,discount_usage_rate,email_open_rate,cart_abandonment_rate,loyalty_score,engagement_score,predicted_churn,churn_probability
25119,25-34,Male,West,Loyal,Online,12,106.85,1462.91,131,67,0.23,0.24,0.57,14,33.5,1,0.616803


In [22]:
sample_customers = X_test.head(10)

predictions = predict_customer_churn(
    sample_customers
)

predictions[
    [
        "predicted_churn",
        "churn_probability"
    ]
]

,predicted_churn,churn_probability
25119,1,6.168026e-01
84952,0,2.415370e-01
37029,0,6.271442e-05
32013,0,6.230200e-05
86566,0,7.849671e-08
89333,1,9.965432e-01
97058,1,9.920109e-01
17984,1,9.863789e-01
92561,0,6.334697e-04
71328,0,3.747535e-05


In [23]:
print(f"Processed data saved to: {processed_data_path.resolve()}")

Processed data saved to: C:\Users\USER\Desktop\AWS_Project\data\processed\processed_customer_churn.csv
